<!-- Здесь проверяем, могут ли исходные модели связно раскрыть тему презентации своими знаниями, без нашего формата. -->
# Разведка: знания и связность исходных моделей

Проверка до DSL, SFT и RL:
- хватает ли модели собственных знаний, чтобы раскрыть тему;
- связно ли она рассуждает от слайда к слайду;
- уместные ли картинки, графики и формулы она предлагает.

Текст печатается прямо в ячейку по мере генерации. В файлы ответы не сохраняются (флаг `SAVE_RESULTS`).

**Запуск.** Run All: модели загружаются по очереди, каждая пишет три презентации и выгружается.
У каждой модели своя ячейка, поэтому выводы разных моделей не затирают друг друга.
Отдельную модель можно перезапустить её ячейкой, но только после всех ячеек до раздела «Загрузка, генерация и выгрузка».

**Kaggle.** Ноутбук самостоятельный: код проекта не нужен, достаточно импортировать этот файл.
В настройках: Accelerator — GPU T4, Internet — On (без интернета модели не скачаются).
Save Version → Save & Run All (Commit) выполняет всё в фоне и сохраняет версию вместе с выводами. На T4 — около часа.

**Локально (GTX 1660 Super, 6 ГБ).** Qwen3-0.6B и 1.7B — без квантования, Qwen3-4B — в 4 битах.
T-lite (8B) в 4 битах — впритык, на длинном ответе может не хватить памяти; её лучше запускать в Kaggle.

Загрузка модели здесь упрощённая, для разведки. Эталонная версия — `src/presentation_rl/model_loader.py` (шаг 8).

## Установка

В Kaggle PyTorch, transformers и accelerate уже есть, ячейка ниже только проверяет это.
Локально на Windows нужен PyTorch с CUDA (обычный `pip install torch` ставит версию без видеокарты):

```
pip install torch --index-url https://download.pytorch.org/whl/cu121
pip install transformers accelerate bitsandbytes
```

Ошибка при импорте transformers про версию `tokenizers` лечится так:
`!pip install -q "tokenizers>=0.22.0,<=0.23.0"` (границы — из текста ошибки), затем перезапуск сессии.

## Настройки

In [11]:
from pathlib import Path

# params_b — число параметров в миллиардах, по нему решается, грузить ли модель в 4 битах.
MODELS = {
    "qwen3_0.6b": {
        "model_id": "Qwen/Qwen3-0.6B", "params_b": 0.6,
        "chat_template_kwargs": {"enable_thinking": False},
        "generation": {"temperature": 0.7, "top_p": 0.8, "top_k": 20},
    },
    "qwen3_1.7b": {
        "model_id": "Qwen/Qwen3-1.7B", "params_b": 1.7,
        "chat_template_kwargs": {"enable_thinking": False},
        "generation": {"temperature": 0.7, "top_p": 0.8, "top_k": 20},
    },
    # Та же 1.7B с режимом рассуждения: сравнить связность с обычным режимом.
    "qwen3_1.7b_thinking": {
        "model_id": "Qwen/Qwen3-1.7B", "params_b": 1.7,
        "chat_template_kwargs": {"enable_thinking": True},
        "generation": {"temperature": 0.6, "top_p": 0.95, "top_k": 20},
    },
    "qwen3_4b": {
        "model_id": "Qwen/Qwen3-4B-Instruct-2507", "params_b": 4.0,
        "chat_template_kwargs": {},
        "generation": {"temperature": 0.7, "top_p": 0.8, "top_k": 20},
    },
    # Для T-lite рекомендован presence_penalty=1.0; в transformers его нет, ближайший аналог — repetition_penalty.
    "tlite_2.1": {
        "model_id": "t-tech/T-lite-it-2.1", "params_b": 8.0,
        "chat_template_kwargs": {},
        "generation": {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "repetition_penalty": 1.05},
    },
}

# Три темы: техническая, естественнонаучная и гуманитарная.
TOPICS = [
    {"id": "gradient_descent", "domain": "technical", "topic": "Градиентный спуск и выбор шага обучения",
     "audience": "студенты второго курса, знакомые с производной"},
    {"id": "photosynthesis", "domain": "technical", "topic": "Фотосинтез: как растения превращают свет в химическую энергию",
     "audience": "школьники 10–11 классов"},
    {"id": "serf_reform", "domain": "humanities", "topic": "Отмена крепостного права в России",
     "audience": "студенты первого курса гуманитарного направления"},
]

MAX_NEW_TOKENS = 4096   # 10–12 слайдов по-русски — примерно 2500–4000 токенов
SEED = 42
FORCE_FP32 = False      # True — если на GTX 1660 в ответе мусор или NaN: fp16 на этой карте иногда сбоит
SAVE_RESULTS = False    # True — дополнительно сохранять ответы в файлы
# В Kaggle писать можно только в /kaggle/working; локально — в runs/ проекта.
IN_KAGGLE = Path("/kaggle/working").exists()
OUT_DIR = Path("/kaggle/working/explore_models") if IN_KAGGLE else Path("../runs/explore_models")

## Промпт

Основной вариант — простая просьба о плане: для каждого слайда текст, изображения и формулы.
Требований к глубине в нём нет намеренно: смотрим, как модель раскрывает тему сама.
Подробный вариант (`DETAILED_PROMPT`) — для сравнения.

In [12]:
SYSTEM_PROMPT = "Ты — опытный преподаватель и автор учебных презентаций. Отвечай на русском языке."

BASIC_PROMPT = (
    "Составь план презентации на тему «{topic}». Аудитория: {audience}.\n"
    "Для каждого слайда укажи, какой текст, изображения и формулы туда вставить."
)

DETAILED_PROMPT = (
    "Составь подробный план презентации на тему «{topic}». Аудитория: {audience}.\n"
    "Презентация должна глубоко раскрыть тему: объяснить механизмы и причины, привести примеры, "
    "показать ограничения и спорные моменты. Сделай 10–12 слайдов. Для каждого слайда укажи:\n"
    "1. Заголовок.\n"
    "2. Текст слайда — связными предложениями, 60–100 слов.\n"
    "3. Какие изображения, графики или схемы вставить и что именно на них показать.\n"
    "4. Формулы в LaTeX, если они уместны для этой темы; если неуместны, напиши «нет»."
)

PROMPT_TEMPLATE = BASIC_PROMPT   # DETAILED_PROMPT — для сравнения


def build_messages(task):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": PROMPT_TEMPLATE.format(**task)},
    ]


print(build_messages(TOPICS[0])[1]["content"])

Составь план презентации на тему «Градиентный спуск и выбор шага обучения». Аудитория: студенты второго курса, знакомые с производной.
Для каждого слайда укажи, какой текст, изображения и формулы туда вставить.


## Устройство

In [13]:
import gc, json, time
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = "xpu"
else:
    DEVICE = "cpu"

if FORCE_FP32 or DEVICE == "cpu":
    DTYPE = torch.float32
elif DEVICE == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8:
    DTYPE = torch.bfloat16  # настоящий bf16 — только с Ampere (RTX 30xx, A100); T4 и P100 его лишь эмулируют
elif DEVICE == "xpu":
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16  # T4, P100, GTX 1660 и другие карты без bf16

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / 1024**3
    print(f"GPU: {props.name}, {VRAM_GB:.1f} ГБ, тип чисел: {DTYPE}")
elif DEVICE == "xpu":
    VRAM_GB = torch.xpu.get_device_properties(0).total_memory / 1024**3
    print(f"XPU: {torch.xpu.get_device_name(0)}, {VRAM_GB:.1f} ГБ, тип чисел: {DTYPE}")
else:
    VRAM_GB = 0.0
    print("Видеокарта не найдена, расчёт на процессоре — очень медленно")

GPU: Tesla T4, 14.6 ГБ, тип чисел: torch.float16


## Загрузка, генерация и выгрузка

`run_model` загружает модель, по очереди пишет презентации на все темы и выгружает модель.
Ошибка одной модели (например, нехватка памяти) печатается в её ячейке и не останавливает следующие.
После каждого ответа печатаются замеры: токены, время, скорость, пик памяти, обрыв по лимиту.

In [14]:
import transformers
from packaging.version import Version
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

# В новых версиях transformers параметр называется dtype, в старых — torch_dtype.
DTYPE_KEY = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"


def load_model(name):
    cfg = MODELS[name]
    # 4 бита — если модель в 16 битах (≈2 байта на параметр) не помещается в 70% видеопамяти.
    load_in_4bit = DEVICE == "cuda" and cfg["params_b"] * 2 > 0.7 * VRAM_GB
    kwargs = {DTYPE_KEY: DTYPE}
    if load_in_4bit:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE
        )
        kwargs["device_map"] = "auto"
    started = time.time()
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"])
    model = AutoModelForCausalLM.from_pretrained(cfg["model_id"], **kwargs)
    if not load_in_4bit:
        model = model.to(DEVICE)
    model.eval()
    print(f"{name} → {cfg['model_id']} | 4 бита: {load_in_4bit} | загрузка {time.time() - started:.0f} с")
    return model, tokenizer, load_in_4bit


def generate(name, model, tokenizer, load_in_4bit, task):
    cfg = MODELS[name]
    inputs = tokenizer.apply_chat_template(
        build_messages(task), add_generation_prompt=True, return_tensors="pt", return_dict=True,
        **cfg["chat_template_kwargs"],
    ).to(model.device)
    torch.manual_seed(SEED)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    started = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, streamer=streamer, **cfg["generation"]
        )
    seconds = time.time() - started
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    meta = {
        "model": name, "model_id": cfg["model_id"], "task": task["id"], "domain": task["domain"],
        "prompt": PROMPT_TEMPLATE.format(**task), "device": DEVICE, "dtype": str(DTYPE), "load_in_4bit": load_in_4bit,
        "seed": SEED, "generation": cfg["generation"], "output_tokens": int(new_tokens.shape[0]),
        "seconds": round(seconds, 1), "tokens_per_second": round(new_tokens.shape[0] / seconds, 1),
        "truncated": int(new_tokens.shape[0]) >= MAX_NEW_TOKENS,
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2) if DEVICE == "cuda" else None,
    }
    return text, meta


def save(text, meta):
    if not SAVE_RESULTS:
        return
    folder = OUT_DIR / meta["model"]
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{meta['task']}.md").write_text(text, encoding="utf-8")
    (folder / f"{meta['task']}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def run_model(name):
    """Загрузка модели, презентации на все темы, выгрузка."""
    model = tokenizer = None
    try:
        model, tokenizer, load_in_4bit = load_model(name)
        for task in TOPICS:
            print(f"\n\n{'=' * 20} {name} | {task['topic']} {'=' * 20}\n")
            text, meta = generate(name, model, tokenizer, load_in_4bit, task)
            save(text, meta)
            print("\n", {k: meta[k] for k in ("output_tokens", "seconds", "tokens_per_second", "truncated", "peak_vram_gb")})
    except Exception as exc:  # одна упавшая модель не останавливает остальные
        print(f"\n!!! {name}: {type(exc).__name__}: {exc}")
    finally:
        del model, tokenizer
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        print(f"\n{name}: память освобождена")

## Qwen3-0.6B

In [15]:
run_model("qwen3_0.6b")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

qwen3_0.6b → Qwen/Qwen3-0.6B | 4 бита: False | загрузка 22 с


==================== qwen3_0.6b | Градиентный спуск и выбор шага обучения ====================

Вот план презентации на тему **«Градиентный спуск и выбор шага обучения»**, адаптированный для студентов второго курса, знакомых с производной:

---

### **Слайд 1: Заголовок**
**Градиентный спуск и выбор шага обучения**

---

### **Слайд 2: Введение**
- **Текст:** "Градиентный спуск — это метод, который позволяет минимизировать функцию в области снизу (или максимизировать вверх) при использовании градиентного спуска."
- **Изображение:** Схема с градиентом.
- **Формула:** $ \nabla f(x) = \frac{\partial f}{\partial x} $

---

### **Слайд 3: Определение градиента**
- **Текст:** Градиент — это вектор, который показывает, какую величину нужно изменить, чтобы уменьшить функцию $ f(x) $.
- **Изображение:** График функции $ f(x) $ с указанием градиента.
- **Формула:** $ \nabla f(x) = \left( \frac{\partial f}{\partial x_1}, \frac{\partia

## Qwen3-1.7B

In [16]:
run_model("qwen3_1.7b")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

qwen3_1.7b → Qwen/Qwen3-1.7B | 4 бита: False | загрузка 6 с


==================== qwen3_1.7b | Градиентный спуск и выбор шага обучения ====================

Конечно! Ниже представлен план презентации на тему **«Градиентный спуск и выбор шага обучения»**, ориентированный на студентов второго курса, знакомых с производной. Каждый слайд включает **текст, изображения и формулы**.

---

### **Слайд 1: Введение**
**Текст:**  
«Градиентный спуск — это метод оптимизации, использующий градиент функции для поиска минимума. Он широко применяется в машинном обучении и оптимизации.»

**Изображение:**  
График функции, показывающий направление градиента.

**Формула:**  
$$
\theta_{k+1} = \theta_k - \alpha \nabla f(\theta_k)
$$  
где:  
- $\theta$ — вектор параметров,  
- $f(\theta)$ — функция ошибки,  
- $\alpha$ — шаг обучения,  
- $\nabla f(\theta)$ — градиент функции.

---

### **Слайд 2: Что такое градиент?**
**Текст:**  
«Градиент — это вектор из частных производных, показывающий направление н

## Qwen3-1.7B с режимом рассуждения

Сначала рассуждение (текст между `<think>` и `</think>`), затем ответ.

In [17]:
run_model("qwen3_1.7b_thinking")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

qwen3_1.7b_thinking → Qwen/Qwen3-1.7B | 4 бита: False | загрузка 6 с


==================== qwen3_1.7b_thinking | Градиентный спуск и выбор шага обучения ====================

<think>
Хорошо, мне нужно составить план презентации на тему "Градиентный спуск и выбор шага обучения" для студентов второго курса, знакомых с производной. Аудитория — это люди, которые уже имеют базовые знания, но могут не быть экспертами. 

Сначала определю структуру презентации. Наверное, начну с введения, затем основные концепции, затем выбор шага обучения, и заключение. Нужно разделить на слайды, убедиться, что каждый слайд содержит текст, изображения и формулы.

Введение: Начну с того, что объясню, что такое градиентный спуск и почему он важен. Может, добавить изображение графика функции, чтобы показать, как спуск происходит. Важно упомянуть, что это метод оптимизации, и что он используется в машинном обучении.

Далее, основные концепции. Здесь нужно объяснить, что такое градиент, как он вычисляется. Нужно 

## Qwen3-4B-Instruct-2507

In [18]:
run_model("qwen3_4b")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

qwen3_4b → Qwen/Qwen3-4B-Instruct-2507 | 4 бита: False | загрузка 64 с


==================== qwen3_4b | Градиентный спуск и выбор шага обучения ====================

Вот подробный и логичный **план презентации** на тему **«Градиентный спуск и выбор шага обучения»** для студентов второго курса, знакомых с производной. Презентация построена так, чтобы быть понятной, наглядной и с акцентом на математическую строгость, но без излишней сложности.

---

### 🎯 **Цель презентации**  
Познакомить студентов с градиентным спуском как методом оптимизации, объяснить роль шага обучения и показать, как его выбор влияет на сходимость и эффективность алгоритма.

---

## 📝 План презентации (12 слайдов)

---

### **Слайд 1: Заголовок**
**Текст:**  
Градиентный спуск и выбор шага обучения  
*Оптимизация функций в машинном обучении*

**Изображение:**  
Иллюстрация поверхности (например, горы или волны), с градиентом, направленным вниз.  
Можно добавить мини-схему: "Градиент → шаг → новая точка".

**Формул

## T-lite-it-2.1

In [19]:
run_model("tlite_2.1")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tlite_2.1 → t-tech/T-lite-it-2.1 | 4 бита: True | загрузка 158 с


==================== tlite_2.1 | Градиентный спуск и выбор шага обучения ====================

Конечно! Ниже — подробный **план презентации** на тему **«Градиентный спуск и выбор шага обучения»**, адаптированный для студентов второго курса, знакомых с производными. Презентация рассчитана на **15–20 минут**, состоит из **8 слайдов** (включая титульный и заключение). Каждый слайд содержит:

- **Текст** (для чтения)
- **Рекомендацию по изображению**
- **Формулы**, если нужны

---

### **Слайд 1: Титульный**

**Текст:**  
**Градиентный спуск и выбор шага обучения**  
*Курс машинного обучения | 2 курс*  
*Преподаватель: [Имя]*  
*Дата: [число]*

**Изображение:**  
Логотип университета или тема в стиле научной визуализации (например, график функции с градиентом).

**Формулы:**  
(нет)

> *Цель слайда — сразу обозначить тему и аудиторию.*

---

### **Слайд 2: Что такое градиентный спуск?**

**Текст:**  
🔹 Градиентный спуск — э

## На что смотреть при чтении

| Вопрос | Признак проблемы |
|---|---|
| Связность | слайды не вытекают друг из друга, повторы, резкие скачки темы |
| Знания | фактические ошибки, выдуманные даты, имена, числа |
| Глубина | только определения и общие слова, нет механизмов, причин, примеров, ограничений |
| Визуал | картинки «для галочки» или то, что невозможно найти или нарисовать |
| Формулы | неверные формулы или формулы там, где они неуместны |
| Язык | английские вставки, иероглифы, обрыв на середине |